In [1]:
import os
import time
import math
import requests
import pandas as pd

# =========================
# 1) SETTINGS
# =========================


API_KEY = "858f32e3199eaf458d2d75c7ed1aa38d"

TARGET_GENRES = ["Action", "Horror", "Comedy", "Romance"]
TARGET_PER_GENRE = 2500

LANGUAGE = "en-US"
ORIGINAL_LANGUAGE = "en"

# نفس الشروط تقريبًا
MIN_RUNTIME = 75
MAX_RUNTIME = 180
MIN_WORDS = 40

START_YEAR = 2000
END_YEAR = 2024
MAX_PAGES_PER_YEAR = 500
SLEEP_SECONDS = 0.12

# حفظ تدريجي
MASTER_PROGRESS_FILE = "tmdb_4genres_progress.csv"
FINAL_OUTPUT_FILE = "tmdb_4genres_2500_each_final.csv"
YEAR_COUNTS_FILE = "tmdb_counts_by_year_and_label_2500.csv"

# الأربع أنواع اللي ممنوع يتداخلوا مع بعض
ALL_TARGET_GENRES = {"Action", "Horror", "Comedy", "Romance"}

# =========================
# 2) API SETUP
# =========================

BASE_URL = "https://api.themoviedb.org/3"
session = requests.Session()

HEADERS = {
    "accept": "application/json",
    "Authorization": f"Bearer {API_KEY}" if API_KEY.startswith("eyJ") else None
}

def build_headers():
    if HEADERS["Authorization"]:
        return {
            "accept": "application/json",
            "Authorization": HEADERS["Authorization"]
        }
    return {"accept": "application/json"}

def call_tmdb(endpoint: str, params: dict | None = None, retries: int = 5) -> dict:
    params = params or {}
    headers = build_headers()

    if not HEADERS["Authorization"]:
        params["api_key"] = API_KEY

    url = f"{BASE_URL}{endpoint}"
    last_error = None

    for attempt in range(retries):
        try:
            response = session.get(url, headers=headers, params=params, timeout=30)
            response.raise_for_status()
            return response.json()
        except Exception as e:
            last_error = e
            print(f"Retry {attempt + 1}/{retries} for {endpoint}: {e}")
            time.sleep(2)

    print(f"Failed permanently for {endpoint}: {last_error}")
    return {}

def get_movie_genres() -> dict:
    data = call_tmdb("/genre/movie/list", {"language": "en"})
    genres = data.get("genres", [])
    return {g["name"]: g["id"] for g in genres}

# =========================
# 3) HELPERS
# =========================

def build_image_url(file_path: str | None, size: str = "w500") -> str | None:
    if not file_path:
        return None
    return f"https://image.tmdb.org/t/p/{size}{file_path}"

def clean_text(text):
    if not isinstance(text, str):
        return ""
    return " ".join(text.replace("\n", " ").replace("\r", " ").split()).strip()

def word_count(text: str) -> int:
    return len(text.split()) if text else 0

def extract_year(release_date: str):
    if not isinstance(release_date, str) or len(release_date) < 4:
        return None
    try:
        return int(release_date[:4])
    except:
        return None

def is_good_candidate(target_label: str, movie_genre_names: list[str], overview: str) -> bool:
    genre_set = set(movie_genre_names)

    # لازم النوع المطلوب موجود
    if target_label not in genre_set:
        return False

    # overview لازم يكون كفاية
    if word_count(overview) < MIN_WORDS:
        return False

    # ممنوع أي نوع تاني من الأربع
    other_target_genres = ALL_TARGET_GENRES - {target_label}
    if genre_set.intersection(other_target_genres):
        return False

    return True

def quality_score(vote_average, vote_count, overview_words):
    vote_count_log = math.log1p(vote_count or 0)
    return (
        0.70 * overview_words +
        0.20 * vote_count_log * 10 +
        0.10 * (vote_average or 0) * 10
    )

# =========================
# 4) PROGRESS HELPERS
# =========================

def load_existing_progress() -> pd.DataFrame:
    if os.path.exists(MASTER_PROGRESS_FILE):
        df = pd.read_csv(MASTER_PROGRESS_FILE, encoding="utf-8-sig")
        print(f"\nLoaded progress file: {MASTER_PROGRESS_FILE}")
        if "label" in df.columns:
            print(df["label"].value_counts(dropna=False))
        return df
    return pd.DataFrame()

def save_progress(df: pd.DataFrame):
    if df.empty:
        return
    df.to_csv(MASTER_PROGRESS_FILE, index=False, encoding="utf-8-sig")
    print(f"Progress saved: {MASTER_PROGRESS_FILE}")

# =========================
# 5) DISCOVER WITH FALLBACK
# =========================

def discover_movies_with_fallback(year: int, genre_id: int, page: int) -> list:
    """
    TMDB أحيانًا يرجع 500 مع بعض ترتيبات sort
    فنجرب أكثر من sort_by
    """
    base_params = {
        "language": LANGUAGE,
        "include_adult": "false",
        "include_video": "false",
        "page": page,
        "with_genres": genre_id,
        "with_original_language": ORIGINAL_LANGUAGE,
        "primary_release_date.gte": f"{year}-01-01",
        "primary_release_date.lte": f"{year}-12-31",
        "with_runtime.gte": MIN_RUNTIME,
        "with_runtime.lte": MAX_RUNTIME,
    }

    sort_options = [
        "popularity.desc",
        "primary_release_date.desc",
        "primary_release_date.asc",
        "vote_average.desc",
    ]

    for sort_value in sort_options:
        params = base_params.copy()
        params["sort_by"] = sort_value

        data = call_tmdb("/discover/movie", params)
        if data and "results" in data:
            results = data.get("results", [])
            return results

    return []

# =========================
# 6) FETCH ONE GENRE
# =========================

def fetch_movies_for_genre_by_year(target_label: str, genre_id: int, existing_df: pd.DataFrame) -> pd.DataFrame:
    print(f"\n==============================")
    print(f"Fetching genre: {target_label}")
    print(f"==============================")

    # حمّل الموجود لهذا النوع لو فيه progress
    if not existing_df.empty and "label" in existing_df.columns:
        current_df = existing_df[existing_df["label"] == target_label].copy()
    else:
        current_df = pd.DataFrame()

    all_rows = current_df.to_dict("records") if not current_df.empty else []

    seen_ids = set()
    if not current_df.empty and "tmdb_id" in current_df.columns:
        seen_ids = set(current_df["tmdb_id"].dropna().astype(int).tolist())

    current_count = len(all_rows)
    print(f"Already have {current_count} for {target_label}")

    if current_count >= TARGET_PER_GENRE:
        print(f"{target_label} already completed.")
        return current_df.head(TARGET_PER_GENRE).copy()

    for year in range(START_YEAR, END_YEAR + 1):
        year_kept_before = len(all_rows)

        for page in range(1, MAX_PAGES_PER_YEAR + 1):
            results = discover_movies_with_fallback(year, genre_id, page)

            if not results:
                break

            for movie in results:
                movie_id = movie.get("id")
                if not movie_id or movie_id in seen_ids:
                    continue

                details = call_tmdb(f"/movie/{movie_id}", {"language": LANGUAGE})
                if not details:
                    continue

                overview = clean_text(details.get("overview") or movie.get("overview") or "")
                genres = details.get("genres", [])
                genre_names = [g["name"] for g in genres if "name" in g]

                if not is_good_candidate(target_label, genre_names, overview):
                    continue

                poster_path = details.get("poster_path")
                backdrop_path = details.get("backdrop_path")

                # لازم poster موجود
                if not poster_path:
                    continue

                release_date = details.get("release_date")
                release_year = extract_year(release_date)

                # تأكيد السنة
                if release_year != year:
                    continue

                overview_words = word_count(overview)

                row = {
                    "tmdb_id": details.get("id"),
                    "title": details.get("title"),
                    "original_title": details.get("original_title"),
                    "release_date": release_date,
                    "release_year": release_year,
                    "original_language": details.get("original_language"),
                    "overview": overview,
                    "text_len_words": overview_words,
                    "genres": ", ".join(genre_names),
                    "genre_ids": [g["id"] for g in genres if "id" in g],
                    "vote_average": details.get("vote_average"),
                    "vote_count": details.get("vote_count"),
                    "popularity": details.get("popularity"),
                    "runtime": details.get("runtime"),
                    "adult": details.get("adult"),
                    "status": details.get("status"),
                    "poster_path": poster_path,
                    "backdrop_path": backdrop_path,
                    "poster_url": build_image_url(poster_path, "w500"),
                    "poster_url_hd": build_image_url(poster_path, "original"),
                    "backdrop_url": build_image_url(backdrop_path, "w780"),
                    "backdrop_url_hd": build_image_url(backdrop_path, "original"),
                    "homepage": details.get("homepage"),
                    "imdb_id": details.get("imdb_id"),
                    "label": target_label,
                }

                row["quality_score"] = quality_score(
                    vote_average=row["vote_average"],
                    vote_count=row["vote_count"],
                    overview_words=row["text_len_words"],
                )

                all_rows.append(row)
                seen_ids.add(movie_id)

                if len(all_rows) >= TARGET_PER_GENRE:
                    break

            if page % 10 == 0:
                print(f"{target_label} | Year {year} | Page {page} | kept={len(all_rows)}")

            if len(all_rows) >= TARGET_PER_GENRE:
                break

            time.sleep(SLEEP_SECONDS)

        year_added = len(all_rows) - year_kept_before
        print(f"{target_label} | Finished year {year}: added {year_added}")

        # حفظ progress بعد كل سنة
        temp_df = pd.DataFrame(all_rows)
        if not temp_df.empty:
            temp_df = temp_df.drop_duplicates(subset=["tmdb_id"]).copy()
            temp_df.to_csv(f"progress_{target_label.lower()}.csv", index=False, encoding="utf-8-sig")

        if len(all_rows) >= TARGET_PER_GENRE:
            break

    df = pd.DataFrame(all_rows)

    if df.empty:
        print(f"No rows found for {target_label}")
        return df

    df = df.drop_duplicates(subset=["tmdb_id"]).copy()

    df = df.sort_values(
        by=["release_year", "quality_score", "text_len_words", "vote_count", "vote_average"],
        ascending=[True, False, False, False, False]
    ).copy()

    df = df.head(TARGET_PER_GENRE).copy()

    print(f"\nFinal kept for {target_label}: {len(df)}")
    return df

# =========================
# 7) MAIN
# =========================

def main():
    genre_map = get_movie_genres()

    missing = [g for g in TARGET_GENRES if g not in genre_map]
    if missing:
        raise ValueError(f"These genres were not found in TMDB genre list: {missing}")

    print("TMDB genre IDs:")
    for g in TARGET_GENRES:
        print(f"{g}: {genre_map[g]}")

    existing_df = load_existing_progress()

    final_parts = []

    for genre_name in TARGET_GENRES:
        genre_id = genre_map[genre_name]
        df_genre = fetch_movies_for_genre_by_year(genre_name, genre_id, existing_df)
        final_parts.append(df_genre)

        # حدّث progress master بعد كل genre
        combined_so_far = pd.concat(final_parts, ignore_index=True)
        if not combined_so_far.empty:
            combined_so_far = combined_so_far.drop_duplicates(subset=["tmdb_id", "label"]).copy()
            save_progress(combined_so_far)

    final_df = pd.concat(final_parts, ignore_index=True)

    if final_df.empty:
        print("No data collected.")
        return

    final_df = final_df.drop_duplicates(subset=["tmdb_id", "label"]).copy()

    print("\n==============================")
    print("Counts by label")
    print("==============================")
    print(final_df["label"].value_counts(dropna=False))

    pivot = pd.pivot_table(
        final_df,
        index="release_year",
        columns="label",
        values="tmdb_id",
        aggfunc="count",
        fill_value=0
    ).sort_index()

    print("\n==============================")
    print("Counts by year and label")
    print("==============================")
    print(pivot)

    pivot.to_csv(YEAR_COUNTS_FILE, encoding="utf-8-sig")
    final_df.to_csv(FINAL_OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\nSaved final dataset: {FINAL_OUTPUT_FILE}")
    print(f"Saved year counts: {YEAR_COUNTS_FILE}")

if __name__ == "__main__":
    main()

TMDB genre IDs:
Action: 28
Horror: 27
Comedy: 35
Romance: 10749

Fetching genre: Action
Already have 0 for Action
Action | Year 2000 | Page 10 | kept=73
Action | Finished year 2000: added 75
Action | Year 2001 | Page 10 | kept=156
Action | Finished year 2001: added 93
Action | Year 2002 | Page 10 | kept=240
Action | Finished year 2002: added 83
Action | Year 2003 | Page 10 | kept=325
Action | Finished year 2003: added 85
Action | Year 2004 | Page 10 | kept=413
Action | Finished year 2004: added 77
Action | Year 2005 | Page 10 | kept=490
Action | Finished year 2005: added 82
Action | Year 2006 | Page 10 | kept=574
Action | Finished year 2006: added 96
Action | Year 2007 | Page 10 | kept=657
Action | Finished year 2007: added 79
Action | Year 2008 | Page 10 | kept=759
Action | Finished year 2008: added 129
Action | Year 2009 | Page 10 | kept=880
Action | Finished year 2009: added 109
Action | Year 2010 | Page 10 | kept=1000
Action | Finished year 2010: added 106
Action | Year 2011 | Page

In [2]:
import os
import time
import math
import requests
import pandas as pd

# =========================
# 1) SETTINGS
# =========================


API_KEY = "858f32e3199eaf458d2d75c7ed1aa38d"

TARGET_LABEL = "Romance"
TARGET_GENRE_ID = 10749
TARGET_TOTAL = 2500

LANGUAGE = "en-US"
ORIGINAL_LANGUAGE = "en"

MIN_RUNTIME = 75
MAX_RUNTIME = 180
MIN_WORDS = 40

START_YEAR = 1970
END_YEAR = 2024
MAX_PAGES_PER_YEAR = 500
SLEEP_SECONDS = 0.12

INPUT_FILE = "tmdb_4genres_2500_each_final.csv"
OUTPUT_FILE = "tmdb_4genres_2500_each_final_updated.csv"
COUNTS_FILE = "tmdb_counts_by_year_and_label_2500_updated.csv"
ROMANCE_PROGRESS_FILE = "progress_romance_extra.csv"

ALL_TARGET_GENRES = {"Action", "Horror", "Comedy", "Romance"}

# =========================
# 2) API SETUP
# =========================

BASE_URL = "https://api.themoviedb.org/3"
session = requests.Session()

HEADERS = {
    "accept": "application/json",
    "Authorization": f"Bearer {API_KEY}" if API_KEY.startswith("eyJ") else None
}

def build_headers():
    if HEADERS["Authorization"]:
        return {
            "accept": "application/json",
            "Authorization": HEADERS["Authorization"]
        }
    return {"accept": "application/json"}

def call_tmdb(endpoint: str, params: dict | None = None, retries: int = 5) -> dict:
    params = params or {}
    headers = build_headers()

    if not HEADERS["Authorization"]:
        params["api_key"] = API_KEY

    url = f"{BASE_URL}{endpoint}"
    last_error = None

    for attempt in range(retries):
        try:
            response = session.get(url, headers=headers, params=params, timeout=30)
            response.raise_for_status()
            return response.json()
        except Exception as e:
            last_error = e
            print(f"Retry {attempt + 1}/{retries} for {endpoint}: {e}")
            time.sleep(2)

    print(f"Failed permanently for {endpoint}: {last_error}")
    return {}

# =========================
# 3) HELPERS
# =========================

def build_image_url(file_path: str | None, size: str = "w500") -> str | None:
    if not file_path:
        return None
    return f"https://image.tmdb.org/t/p/{size}{file_path}"

def clean_text(text):
    if not isinstance(text, str):
        return ""
    return " ".join(text.replace("\n", " ").replace("\r", " ").split()).strip()

def word_count(text: str) -> int:
    return len(text.split()) if text else 0

def extract_year(release_date: str):
    if not isinstance(release_date, str) or len(release_date) < 4:
        return None
    try:
        return int(release_date[:4])
    except:
        return None

def is_good_candidate(target_label: str, movie_genre_names: list[str], overview: str) -> bool:
    genre_set = set(movie_genre_names)

    if target_label not in genre_set:
        return False

    if word_count(overview) < MIN_WORDS:
        return False

    other_target_genres = ALL_TARGET_GENRES - {target_label}
    if genre_set.intersection(other_target_genres):
        return False

    return True

def quality_score(vote_average, vote_count, overview_words):
    vote_count_log = math.log1p(vote_count or 0)
    return (
        0.70 * overview_words +
        0.20 * vote_count_log * 10 +
        0.10 * (vote_average or 0) * 10
    )

# =========================
# 4) DISCOVER WITH FALLBACK
# =========================

def discover_movies_with_fallback(year: int, genre_id: int, page: int) -> list:
    base_params = {
        "language": LANGUAGE,
        "include_adult": "false",
        "include_video": "false",
        "page": page,
        "with_genres": genre_id,
        "with_original_language": ORIGINAL_LANGUAGE,
        "primary_release_date.gte": f"{year}-01-01",
        "primary_release_date.lte": f"{year}-12-31",
        "with_runtime.gte": MIN_RUNTIME,
        "with_runtime.lte": MAX_RUNTIME,
    }

    sort_options = [
        "popularity.desc",
        "primary_release_date.desc",
        "primary_release_date.asc",
        "vote_average.desc",
    ]

    for sort_value in sort_options:
        params = base_params.copy()
        params["sort_by"] = sort_value

        data = call_tmdb("/discover/movie", params)
        if data and "results" in data:
            return data.get("results", [])

    return []

# =========================
# 5) LOAD EXISTING DATA
# =========================

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

existing_df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")

if "label" not in existing_df.columns:
    raise ValueError("Input file must contain a 'label' column")

print("Current counts:")
print(existing_df["label"].value_counts(dropna=False))

romance_df = existing_df[existing_df["label"] == TARGET_LABEL].copy()
other_df = existing_df[existing_df["label"] != TARGET_LABEL].copy()

current_romance_count = len(romance_df)
print(f"\nCurrent Romance count = {current_romance_count}")

if current_romance_count >= TARGET_TOTAL:
    print("Romance already reached target.")
    existing_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"Saved unchanged file: {OUTPUT_FILE}")
    raise SystemExit

needed = TARGET_TOTAL - current_romance_count
print(f"Need extra Romance movies = {needed}")

# كل IDs الموجودة بالفعل في كل الداتا
existing_all_ids = set(existing_df["tmdb_id"].dropna().astype(int).tolist())

# كل IDs الموجودة في Romance الحالي
existing_romance_ids = set(romance_df["tmdb_id"].dropna().astype(int).tolist())

# =========================
# 6) FETCH EXTRA ROMANCE
# =========================

new_rows = []

for year in range(START_YEAR, END_YEAR + 1):
    year_added_before = len(new_rows)

    for page in range(1, MAX_PAGES_PER_YEAR + 1):
        results = discover_movies_with_fallback(year, TARGET_GENRE_ID, page)

        if not results:
            break

        for movie in results:
            movie_id = movie.get("id")
            if not movie_id:
                continue

            # ما نكررهوش لا في الملف الحالي ولا في اللي بنضيفه
            if movie_id in existing_all_ids or movie_id in existing_romance_ids:
                continue

            details = call_tmdb(f"/movie/{movie_id}", {"language": LANGUAGE})
            if not details:
                continue

            overview = clean_text(details.get("overview") or movie.get("overview") or "")
            genres = details.get("genres", [])
            genre_names = [g["name"] for g in genres if "name" in g]

            if not is_good_candidate(TARGET_LABEL, genre_names, overview):
                continue

            poster_path = details.get("poster_path")
            backdrop_path = details.get("backdrop_path")

            if not poster_path:
                continue

            release_date = details.get("release_date")
            release_year = extract_year(release_date)

            if release_year != year:
                continue

            overview_words = word_count(overview)

            row = {
                "tmdb_id": details.get("id"),
                "title": details.get("title"),
                "original_title": details.get("original_title"),
                "release_date": release_date,
                "release_year": release_year,
                "original_language": details.get("original_language"),
                "overview": overview,
                "text_len_words": overview_words,
                "genres": ", ".join(genre_names),
                "genre_ids": [g["id"] for g in genres if "id" in g],
                "vote_average": details.get("vote_average"),
                "vote_count": details.get("vote_count"),
                "popularity": details.get("popularity"),
                "runtime": details.get("runtime"),
                "adult": details.get("adult"),
                "status": details.get("status"),
                "poster_path": poster_path,
                "backdrop_path": backdrop_path,
                "poster_url": build_image_url(poster_path, "w500"),
                "poster_url_hd": build_image_url(poster_path, "original"),
                "backdrop_url": build_image_url(backdrop_path, "w780"),
                "backdrop_url_hd": build_image_url(backdrop_path, "original"),
                "homepage": details.get("homepage"),
                "imdb_id": details.get("imdb_id"),
                "label": TARGET_LABEL,
            }

            row["quality_score"] = quality_score(
                vote_average=row["vote_average"],
                vote_count=row["vote_count"],
                overview_words=row["text_len_words"],
            )

            new_rows.append(row)
            existing_all_ids.add(movie_id)
            existing_romance_ids.add(movie_id)

            if current_romance_count + len(new_rows) >= TARGET_TOTAL:
                break

        if page % 10 == 0:
            print(f"Romance extra | Year {year} | Page {page} | added so far = {len(new_rows)}")

        if current_romance_count + len(new_rows) >= TARGET_TOTAL:
            break

        time.sleep(SLEEP_SECONDS)

    year_added = len(new_rows) - year_added_before
    print(f"Romance extra | Finished year {year}: added {year_added}")

    # حفظ progress للرومانسي الإضافي
    if new_rows:
        temp_extra_df = pd.DataFrame(new_rows)
        temp_extra_df = temp_extra_df.drop_duplicates(subset=["tmdb_id"]).copy()
        temp_extra_df.to_csv(ROMANCE_PROGRESS_FILE, index=False, encoding="utf-8-sig")

    if current_romance_count + len(new_rows) >= TARGET_TOTAL:
        break

# =========================
# 7) MERGE & SAVE
# =========================

extra_romance_df = pd.DataFrame(new_rows)

if not extra_romance_df.empty:
    extra_romance_df = extra_romance_df.drop_duplicates(subset=["tmdb_id"]).copy()

    combined_romance = pd.concat([romance_df, extra_romance_df], ignore_index=True)
    combined_romance = combined_romance.drop_duplicates(subset=["tmdb_id"]).copy()

    combined_romance = combined_romance.sort_values(
        by=["release_year", "quality_score", "text_len_words", "vote_count", "vote_average"],
        ascending=[True, False, False, False, False]
    ).copy()

    combined_romance = combined_romance.head(TARGET_TOTAL).copy()
else:
    combined_romance = romance_df.copy()

final_df = pd.concat([other_df, combined_romance], ignore_index=True)
final_df = final_df.drop_duplicates(subset=["tmdb_id", "label"]).copy()

print("\nFinal counts:")
print(final_df["label"].value_counts(dropna=False))

pivot = pd.pivot_table(
    final_df,
    index="release_year",
    columns="label",
    values="tmdb_id",
    aggfunc="count",
    fill_value=0
).sort_index()

print("\nCounts by year and label:")
print(pivot)

final_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
pivot.to_csv(COUNTS_FILE, encoding="utf-8-sig")

print(f"\nSaved updated dataset: {OUTPUT_FILE}")
print(f"Saved updated counts: {COUNTS_FILE}")

Current counts:
label
Action     2500
Horror     2500
Comedy     2500
Romance    1964
Name: count, dtype: int64

Current Romance count = 1964
Need extra Romance movies = 536
Romance extra | Finished year 1970: added 14
Romance extra | Finished year 1971: added 9
Romance extra | Finished year 1972: added 4
Romance extra | Finished year 1973: added 10
Romance extra | Finished year 1974: added 10
Romance extra | Finished year 1975: added 7
Romance extra | Finished year 1976: added 5
Romance extra | Finished year 1977: added 6
Romance extra | Finished year 1978: added 10
Romance extra | Finished year 1979: added 12
Romance extra | Finished year 1980: added 13
Romance extra | Finished year 1981: added 15
Romance extra | Finished year 1982: added 26
Romance extra | Finished year 1983: added 16
Romance extra | Finished year 1984: added 20
Romance extra | Finished year 1985: added 12
Romance extra | Finished year 1986: added 23
Romance extra | Finished year 1987: added 25
Romance extra | Finis